# Notebook 1: Synthetic Experiment (Section 5.1)

Validates the identifiability theory from the paper.
- No LLM required
- Generates synthetic data with known latent structure (Laplace + linear mixing)
- Recovers latents using ICA (FastICA / Picard)
- Compares with AE+Jacobian L1 baseline
- Reproduces Fig 3 (R² matrix) and Fig 4 (MCC across dimensions)

**Key finding**: AE+L1 fails at dim≥128 (MCC≈0.21). ICA achieves MCC>0.97 up to dim=384.

**Estimated time: ~5 minutes on Colab (CPU sufficient)**

In [ ]:
# Setup: Clone repo and install
# To push results back to GitHub, store a Personal Access Token in Colab Secrets:
#   1. Go to https://github.com/settings/tokens → Generate new token (classic) → repo scope
#   2. In Colab left sidebar → Secrets (key icon) → Add: name="GITHUB_TOKEN", value=<your token>
#   3. Toggle "Notebook access" ON

import os
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/AUMEZAK/thoughtcomm.git'
except Exception:
    GITHUB_TOKEN = None
    REPO_URL = 'https://github.com/AUMEZAK/thoughtcomm.git'

!git clone {REPO_URL} thoughtcomm 2>/dev/null || echo 'Already cloned'
%cd thoughtcomm
!pip install -e . -q
!pip install python-picard -q

# Configure git for pushing results
!git config user.email "colab@thoughtcomm.dev"
!git config user.name "ThoughtComm Colab"

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
from data.synthetic import generate_synthetic_data
from training.ica_solver import run_ica, run_fastica, run_picard, HAS_PICARD
from evaluation.synthetic_eval import (
    compute_r2_matrix, compute_mcc_fast,
    recover_b_matrix, evaluate_b_matrix,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Picard available: {HAS_PICARD}')

## 1. Basic Setup: R² Matrix (Fig 3)

In [ ]:
# Generate synthetic data (dim=128, linear mixing)
DIM = 128
N_SAMPLES = max(10000, DIM * 200)  # More samples for better ICA convergence

X, Z, B_true, group_indices, mixing_fn = generate_synthetic_data(
    dim=DIM, num_samples=N_SAMPLES, seed=42, num_layers=1  # linear mixing for ICA
)
print(f'X shape: {X.shape}, Z shape: {Z.shape}')
print(f'Group indices: {group_indices}')
print(f'B_true non-zero fraction: {B_true.float().mean():.3f}')
print(f'Samples: {N_SAMPLES}')

In [ ]:
# Run ICA (auto-selects FastICA or Picard based on dimension)
t0 = time.time()
Z_hat_ica, ica_model, ica_info = run_ica(X.numpy(), verbose=True)
t_ica = time.time() - t0

# Compute MCC
Z_hat_t = torch.tensor(Z_hat_ica, dtype=torch.float32)
mcc_ica, perm_ica = compute_mcc_fast(Z_hat_t, Z)
print(f'\nICA: MCC={mcc_ica:.4f} ({t_ica:.1f}s)')
print(f'  Method: {ica_info["method"]}')
print(f'  Converged: {ica_info.get("converged", "N/A")}')

In [ ]:
# Also train AE+L1 and baseline AE for comparison (expect MCC~0.21 at dim=128)
from configs.config import ThoughtCommConfig
from models.autoencoder import SparsityRegularizedAE
from training.train_autoencoder import train_autoencoder, train_autoencoder_baseline

config = ThoughtCommConfig(
    n_z=DIM, ae_hidden=256, ae_num_layers=3,
    ae_epochs=200, ae_batch_size=128, ae_lr=1e-3,
    jacobian_l1_weight=0.5, jacobian_sample_rows=32,
    device=device,
)
config.hidden_size = DIM // 2
config.num_agents = 2

print('Training AE+L1 (sparsity-regularized)...')
ae_ours, loss_ours, norm_stats = train_autoencoder(X, config, verbose=True)

print('\nTraining baseline AE (no sparsity, same normalization)...')
ae_base, loss_base, _ = train_autoencoder_baseline(X, config, verbose=True, norm_stats=norm_stats)

In [ ]:
# Compute MCC for AE methods
# encode() auto-normalizes (norm_stats embedded in model after training)
X_f = X.float().to(device)

with torch.no_grad():
    Z_hat_ae = ae_ours.encode(X_f).cpu()
    Z_hat_base = ae_base.encode(X_f).cpu()

mcc_ae, _ = compute_mcc_fast(Z_hat_ae, Z)
mcc_base, _ = compute_mcc_fast(Z_hat_base, Z)

print(f'ICA:      MCC = {mcc_ica:.4f}')
print(f'AE+L1:    MCC = {mcc_ae:.4f}')
print(f'Baseline: MCC = {mcc_base:.4f}')

In [ ]:
# Compute R² matrices for all three methods
# Align ICA latents to true Z ordering via permutation
Z_hat_ica_aligned = Z_hat_t[:, perm_ica]

r2_ica, names = compute_r2_matrix(Z_hat_ica_aligned, Z, group_indices)
r2_ae, _ = compute_r2_matrix(Z_hat_ae, Z, group_indices)
r2_base, _ = compute_r2_matrix(Z_hat_base, Z, group_indices)

print('ICA R² matrix:')
print(np.round(r2_ica, 3))
print('\nAE+L1 R² matrix:')
print(np.round(r2_ae, 3))
print('\nBaseline R² matrix:')
print(np.round(r2_base, 3))

In [ ]:
# Plot Fig 3: R² matrices (ICA vs AE+L1 vs Baseline)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
labels = ['$Z_A \\setminus Z_B$', '$Z_A \\cap Z_B$', '$Z_B \\setminus Z_A$']

for ax, r2, title in [
    (axes[0], r2_ica, f'ICA (MCC={mcc_ica:.2f})'),
    (axes[1], r2_ae, f'AE+L1 (MCC={mcc_ae:.2f})'),
    (axes[2], r2_base, f'Baseline (MCC={mcc_base:.2f})')
]:
    im = ax.imshow(r2, cmap='Blues', vmin=0, vmax=1.0)
    ax.set_title(title, fontsize=13)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{r2[i,j]:.2f}', ha='center', va='center', fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle(f'Figure 3: R² of estimated vs true latent groups (dim={DIM})', fontsize=14)
plt.tight_layout()
plt.savefig('fig3_r2_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. MCC Across Dimensions (Fig 4)

ICA-based latent recovery. Uses FastICA for dim<=256, Picard for dim>256.

In [ ]:
# MCC sweep across dimensions with ICA
dims = [128, 256, 384, 512]
mcc_ica_results = []
mcc_ae_results = []
times_ica = []

for dim in dims:
    n_samples = max(10000, dim * 200)
    print(f'\nDimension: {dim} (n_samples={n_samples})')

    X_d, Z_d, B_d, gi_d, _ = generate_synthetic_data(
        dim=dim, num_samples=n_samples, seed=42, num_layers=1
    )

    # ICA
    t0 = time.time()
    Z_hat_d, _, info_d = run_ica(X_d.numpy(), verbose=True)
    t_d = time.time() - t0
    times_ica.append(t_d)

    if Z_hat_d is not None:
        Z_ht = torch.tensor(Z_hat_d, dtype=torch.float32)
        mcc_d, _ = compute_mcc_fast(Z_ht, Z_d)
    else:
        mcc_d = 0.0
    mcc_ica_results.append(float(mcc_d))
    print(f'  ICA: MCC={mcc_d:.4f} ({t_d:.1f}s) method={info_d["method"]}')

    # AE+L1 for comparison (quick, 100 epochs)
    cfg = ThoughtCommConfig(
        n_z=dim, ae_hidden=256, ae_num_layers=3,
        ae_epochs=100, ae_batch_size=128, ae_lr=1e-3,
        jacobian_l1_weight=0.5, jacobian_sample_rows=32, device=device,
    )
    cfg.hidden_size = dim // 2
    cfg.num_agents = 2

    ae_d, _, ns_d = train_autoencoder(X_d, cfg, verbose=False)
    # encode() auto-normalizes (norm_stats embedded in model after training)
    X_df = X_d.float().to(device)
    with torch.no_grad():
        Z_ae_d = ae_d.encode(X_df).cpu()
    mcc_ae_d, _ = compute_mcc_fast(Z_ae_d, Z_d)
    mcc_ae_results.append(float(mcc_ae_d))
    print(f'  AE+L1: MCC={mcc_ae_d:.4f}')

print('\n--- Summary ---')
for i, dim in enumerate(dims):
    print(f'  dim={dim}: ICA={mcc_ica_results[i]:.4f}, AE+L1={mcc_ae_results[i]:.4f}')

In [ ]:
# Plot Fig 4: MCC across dimensions (ICA vs AE+L1)
plt.figure(figsize=(8, 5))
plt.plot(dims, mcc_ica_results, 'b-o', linewidth=2, markersize=8, label='ICA')
plt.plot(dims, mcc_ae_results, 'g-s', linewidth=2, markersize=8, label='AE+L1')
plt.axhline(y=0.75, color='r', linestyle='--', alpha=0.7, label='Identifiability threshold')
plt.xlabel('Dimension', fontsize=12)
plt.ylabel('MCC', fontsize=12)
plt.title('Figure 4: MCC across setups', fontsize=14)
plt.legend(fontsize=11)
plt.xticks(dims)
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.savefig('fig4_mcc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# B matrix recovery from ICA (agent-aware method)
n_obs_a = DIM // 2
B_est_ica, scores_ica = recover_b_matrix(
    X.numpy(), Z_hat_ica, n_obs_a, perm=perm_ica, method='agent_aware'
)
b_metrics = evaluate_b_matrix(B_est_ica, B_true)
print(f'B matrix recovery (agent-aware):')
print(f'  Accuracy={b_metrics["accuracy"]:.4f}, Precision={b_metrics["precision"]:.4f}')
print(f'  Recall={b_metrics["recall"]:.4f}, F1={b_metrics["f1"]:.4f}')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(B_true.numpy(), cmap='Blues', aspect='auto')
axes[0].set_title('Ground Truth B', fontsize=12)
axes[0].set_xlabel('Latent dims')
axes[0].set_ylabel('Observed dims')

axes[1].imshow(B_est_ica, cmap='Blues', aspect='auto')
axes[1].set_title(f'Estimated B (ICA, F1={b_metrics["f1"]:.2f})', fontsize=12)
axes[1].set_xlabel('Latent dims')
axes[1].set_ylabel('Observed dims')

plt.tight_layout()
plt.savefig('b_matrix_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

**Results:**
- **ICA** achieves MCC > 0.97 for dim <= 384 (perfect latent recovery)
- **AE+L1** gives MCC ~ 0.21 regardless of lambda (complete failure at dim=128+)
- **R² matrix (ICA)**: Strong diagonal structure (high R² on matching groups, low cross)
- **R² matrix (AE+L1/Baseline)**: No disentanglement

**Key insight from investigation (17 experiments across 4 sessions):**
- AE + Jacobian L1 is fundamentally broken for dim >= 128
- L1 uniformly shrinks all Jacobian entries (separation ratio = 1.0x)
- Even ICA-perfect initialization is destroyed by AE training (MCC 0.99 -> 0.24)
- ICA works because Laplace + linear mixing satisfies classical ICA conditions

## Push Results to GitHub

In [ ]:
# Save results and push to GitHub
import json

results_01 = {
    'r2_ica': r2_ica.tolist(),
    'r2_ae_l1': r2_ae.tolist(),
    'r2_baseline': r2_base.tolist(),
    'mcc_dims': dims,
    'mcc_ica': mcc_ica_results,
    'mcc_ae_l1': mcc_ae_results,
    'ica_times': times_ica,
    'b_matrix_f1': b_metrics['f1'],
    'b_true_sparsity': float(1 - B_true.float().mean()),
}

os.makedirs('results', exist_ok=True)
with open('results/01_synthetic_results.json', 'w') as f:
    json.dump(results_01, f, indent=2)

# Copy figures to results/
!cp fig3_r2_matrix.png results/
!cp fig4_mcc.png results/
!cp b_matrix_comparison.png results/

# Git add, commit, push
!git add results/
!git commit -m "Add Notebook 01 results: synthetic experiment with ICA"
!git push

print('Results pushed to GitHub!')